# Agent 6 - Career Agent

Run every cell from top to bottom. This notebook implements the framework's
`Agent 3 -> Agent 6` handoff. The student has no career-outcomes dataset, so the active
implementation is an explicit **rule-based advisory fallback**. A separate supervised
training and frozen evaluation path is included but is not run on generated labels.

**Input:** `state["programs"]` from Agent 3 and `state["profile"]`, preferably with
`career_goal="research"` or `career_goal="industry"`. Program descriptions come from
`data/raw/agent2/agent2_enhanced_program_catalog.csv`.

**Output:** copied state with `state["careers"]` and `status="career_done"`. Each entry
contains `university`, `program_name`, `career_path`, `alignment_score`, `expected_roles`,
and `avg_placement_stats`, plus evidence, method, provenance and review flags.
Unknown/mixed evidence produces `career_path=None`; missing goals produce
`alignment_score=None`. No candidate is silently discarded. Admission probability
and tier are not required; Agent 3's unknown values are tolerated.

An advisory alignment score is the share of distinct positive program cues supporting
the student's goal. It is not accuracy, admission likelihood, employment probability,
or a calibrated score. Role examples are possible directions to investigate, not job offers.
No salaries, visa eligibility, or placement rates are inferred from occupation descriptions.

## 1. Local setup and uploaded program catalog

In [1]:
from pathlib import Path
from copy import deepcopy
import hashlib
import json
import math
import re
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/raw').is_dir())
AGENT6_DATA = PROJECT_ROOT / 'data/raw/agent6'
AGENT6_CATALOG_PATH = PROJECT_ROOT / 'data/raw/agent2/agent2_enhanced_program_catalog.csv'
AGENT6_OUTCOMES_PATH = AGENT6_DATA / 'agent6_career_outcomes.csv'
AGENT6_PLACEMENTS_PATH = AGENT6_DATA / 'agent6_placement_reports.csv'
AGENT6_RUNS = AGENT6_DATA / 'evaluation_runs'

def career_text(value):
    if value is None:
        return ''
    if isinstance(value, (list, tuple)):
        return ' '.join(career_text(item) for item in value)
    if isinstance(value, dict):
        return ''
    return '' if pd.isna(value) else str(value).strip()

def career_key(value):
    return ' '.join(career_text(value).casefold().split())

def load_career_catalog(path=AGENT6_CATALOG_PATH):
    frame = pd.read_csv(path).fillna('')
    required = {'university_name', 'program_name', 'description_text', 'curriculum_highlights', 'faculty_research_areas'}
    if required - set(frame):
        raise ValueError('Program catalog is missing descriptive fields.')
    frame['_university_key'] = frame.university_name.map(career_key)
    frame['_program_key'] = frame.program_name.map(career_key)
    if frame[['_university_key', '_program_key']].eq('').any().any():
        raise ValueError('Every catalog row needs a named university and program.')
    # Prefer the richest descriptive duplicate, as in Agent 3.
    frame['_length'] = frame[['description_text', 'curriculum_highlights', 'faculty_research_areas']].agg(' '.join, axis=1).str.len()
    return frame.sort_values('_length', ascending=False, kind='stable').drop_duplicates(
        ['_university_key', '_program_key']).reset_index(drop=True)

career_catalog = load_career_catalog()
print(f'Agent 6 catalog: {len(career_catalog):,} programs; {career_catalog.university_name.nunique()} universities.')
print('No career-outcome dataset was supplied. Active method: transparent advisory rules.')

Agent 6 catalog: 3,052 programs; 50 universities.
No career-outcome dataset was supplied. Active method: transparent advisory rules.


## 2. Explainable fallback and sourced role examples

Research evidence includes thesis/dissertation/research/doctoral language. Industry
evidence includes internship, professional projects, deployment and applied design.
The fallback honors a stated goal only where program text supports it; otherwise it
shows the supported path or abstains. These are transparent heuristics, not a trained model.
Negation handling covers simple clauses; complex or mixed goals should be explicit.

Role titles are grounded in the U.S. Bureau of Labor Statistics occupational categories,
reviewed 2026-10-21. The program-to-role mapping is our advisory inference, not a BLS
endorsement or evidence of a particular university's graduate placements:
[Computing](https://www.bls.gov/ooh/computer-and-information-technology/home.htm),
[Engineering](https://www.bls.gov/ooh/architecture-and-engineering/home.htm),
[Math](https://www.bls.gov/ooh/math/home.htm),
[Business](https://www.bls.gov/ooh/business-and-financial/home.htm),
[Computing research](https://www.bls.gov/ooh/computer-and-information-technology/computer-and-information-research-scientists.htm).
Graduate-researcher suggestions outside computing are generic study directions from
program evidence, without an asserted occupational source or guaranteed position.

`avg_placement_stats` stays null unless a report matches the exact university/program,
country and dated cohort with valid surveyed/placed counts and source provenance.
The field holds that report's rate and definition, never an average of incompatible cohorts.

In [2]:
# Evidence rules are authored heuristics, never training labels or measured success probabilities.
RESEARCH_CUES = ('thesis', 'dissertation', 'research', 'doctoral', 'phd', 'experiments')
INDUSTRY_CUES = ('internship', 'industry', 'professional', 'capstone', 'deployment',
                 'manufacturing', 'software', 'design', 'management', 'engineering')
GOAL_RESEARCH = ('research', 'researcher', 'scientist', 'phd', 'doctoral', 'academia', 'professor')
GOAL_INDUSTRY = ('industry', 'job', 'employment', 'professional', 'developer', 'engineer', 'consulting', 'entrepreneur')

def positive_mentions(text, vocabulary):
    normalized = re.sub(r"[^a-z0-9\s.!?;,-]", ' ', career_text(text).casefold())
    found = []
    for phrase in vocabulary:
        for match in re.finditer(r'\b' + re.escape(phrase) + r'\b', normalized):
            # Negation in the same short clause suppresses the cue. This is conservative,
            # not a full language parser; ambiguous goals request explicit research/industry.
            clause = re.split(r'[.!?;,]|\b(?:but|however)\b', normalized[:match.start()])[-1]
            if not re.search(r'\b(?:no|not|never|without|avoid|neither)\b', ' '.join(clause.split()[-5:])):
                found.append(phrase)
                break
    return sorted(set(found))

def stated_career_goal(profile):
    explicit = career_key(profile.get('career_goal'))
    aliases = {'research': 'research', 'research-track': 'research', 'academia': 'research',
               'phd': 'research', 'industry': 'industry', 'job': 'industry', 'job-track': 'industry'}
    if explicit in aliases:
        return aliases[explicit], 'explicit career_goal'
    # Interests or academic merit are not silently treated as career intentions.
    text = career_text(profile.get('career_goal')) or career_text(profile.get('career_goals'))
    research, industry = positive_mentions(text, GOAL_RESEARCH), positive_mentions(text, GOAL_INDUSTRY)
    if bool(research) != bool(industry):
        return ('research' if research else 'industry'), 'interpreted free-text goal; confirm with student'
    return None, 'missing or mixed goal; set profile.career_goal to research or industry'

BLS_ENGINEERING = 'https://www.bls.gov/ooh/architecture-and-engineering/home.htm'
BLS_COMPUTING = 'https://www.bls.gov/ooh/computer-and-information-technology/home.htm'
BLS_MATH = 'https://www.bls.gov/ooh/math/home.htm'
BLS_BUSINESS = 'https://www.bls.gov/ooh/business-and-financial/home.htm'
# Program-to-occupation mappings are advisory inferences, not measured graduate destinations.
ROLE_FAMILIES = [
    ('cybersecurity', ('cybersecurity',), ['Information security analyst'], BLS_COMPUTING),
    ('data science', ('data science', 'artificial intelligence'), ['Data scientist', 'Software developer'], BLS_COMPUTING),
    ('statistics', ('statistics',), ['Statistician', 'Data scientist'], BLS_MATH),
    ('applied mathematics', ('applied mathematics',), ['Operations research analyst', 'Mathematician'], BLS_MATH),
    ('business analytics', ('business analytics',), ['Operations research analyst', 'Management analyst'], BLS_BUSINESS),
    ('computer engineering', ('computer engineering',), ['Computer hardware engineer'], BLS_ENGINEERING),
    ('electrical engineering', ('electrical engineering',), ['Electrical engineer', 'Electronics engineer'], BLS_ENGINEERING),
    ('biomedical engineering', ('biomedical engineering',), ['Biomedical engineer'], BLS_ENGINEERING),
    ('chemical engineering', ('chemical engineering',), ['Chemical engineer'], BLS_ENGINEERING),
    ('civil engineering', ('civil engineering',), ['Civil engineer'], BLS_ENGINEERING),
    ('industrial engineering', ('industrial engineering',), ['Industrial engineer'], BLS_ENGINEERING),
    ('mechanical engineering', ('mechanical engineering', 'robotics'), ['Mechanical engineer'], BLS_ENGINEERING),
    ('information systems', ('information systems',), ['Computer systems analyst'], BLS_COMPUTING),
    ('business administration', ('business administration',), ['Management analyst'], BLS_BUSINESS),
    ('computing', ('computer science', 'software engineering'), ['Software developer'], BLS_COMPUTING),
]

def suggested_roles(program_name, path):
    for family, phrases, industry_roles, source in ROLE_FAMILIES:
        if any(phrase in career_key(program_name) for phrase in phrases):
            if path == 'research':
                if family in {'computing', 'data science', 'cybersecurity', 'computer engineering', 'information systems'}:
                    return ['Computer and information research scientist'], [
                        'https://www.bls.gov/ooh/computer-and-information-technology/computer-and-information-research-scientists.htm']
                return ['Graduate researcher in ' + family], []
            if path == 'industry':
                return industry_roles, list(dict.fromkeys([source] + ([BLS_MATH] if family in {'data science', 'business analytics'} else [])))
    return [], []

PLACEMENT_COLUMNS = ['university_name', 'program_name', 'cohort_year', 'country', 'definition',
                     'placed_count', 'surveyed_count', 'source_url', 'verified_on']

def placement_for_program(reports, university, program, country, as_of):
    if reports is None or reports.empty:
        return None, 'No program-specific placement report supplied.'
    if set(PLACEMENT_COLUMNS) - set(reports):
        raise ValueError('Placement reports are missing required columns.')
    candidates = reports.loc[reports.university_name.map(career_key).eq(career_key(university))
        & reports.program_name.map(career_key).eq(career_key(program))].copy()
    if candidates.empty:
        return None, 'No report matches this exact university/program.'
    if candidates.duplicated(['university_name', 'program_name', 'cohort_year']).any():
        raise ValueError('Reconcile duplicate reports for the same program/cohort.')
    valid = []
    for row in candidates.to_dict('records'):
        try:
            placed, surveyed, year = float(row['placed_count']), float(row['surveyed_count']), float(row['cohort_year'])
            verified = pd.Timestamp(row['verified_on'])
        except (TypeError, ValueError):
            continue
        if (not all(math.isfinite(x) and x.is_integer() for x in (placed, surveyed, year))
                or not (0 <= placed <= surveyed and surveyed > 0 and 1900 <= year <= as_of.year)
                or pd.isna(verified) or verified.date() > as_of
                or not career_text(row['source_url']).startswith('https://')
                or not career_text(row['definition']) or not career_key(country)
                or career_key(row['country']) != career_key(country)):
            continue
        valid.append({**row, 'cohort_year': int(year), 'placed_count': int(placed), 'surveyed_count': int(surveyed),
                      'placement_rate': placed / surveyed})
    if not valid:
        return None, 'Matching report has invalid counts, date, country or provenance; statistics withheld.'
    latest = max(valid, key=lambda row: row['cohort_year'])
    return {key: latest[key] for key in PLACEMENT_COLUMNS + ['placement_rate']}, 'Source-specific reported cohort rate; not a forecast.'

def career_features(profile, program, catalog_row):
    # Identical feature construction is used in training and inference.
    goal = career_text(profile.get('career_goal')) or career_text(profile.get('career_goals'))
    return '\n'.join([career_text(program.get('program_name')), career_text(catalog_row.get('description_text')),
        career_text(catalog_row.get('curriculum_highlights')), career_text(catalog_row.get('faculty_research_areas')),
        career_text(profile.get('interests_text')), goal])

def career_agent(state, catalog=None, *, model_bundle=None, placement_reports=None, as_of=None):
    """Consume Agent 3 state['programs']; preserve its order and write state['careers']."""
    catalog = career_catalog if catalog is None else catalog
    if {'_university_key', '_program_key'} - set(catalog):
        raise ValueError('Pass a catalog prepared by load_career_catalog.')
    lookup = {(row['_university_key'], row['_program_key']): row for row in catalog.to_dict('records')}
    profile, rows = state.get('profile') or {}, []
    goal, goal_note = stated_career_goal(profile)
    as_of = pd.Timestamp.now(tz='Asia/Kolkata').date() if as_of is None else pd.Timestamp(as_of).date()
    if pd.isna(as_of):
        raise ValueError('as_of must be a valid date.')
    if model_bundle is not None and not model_bundle.get('independent_labels_confirmed'):
        raise ValueError('Model must come from verified independent outcome labels.')
    for program in state.get('programs') or []:
        university = career_text(program.get('university')) or career_text(program.get('university_name'))
        name = career_text(program.get('program_name'))
        found = lookup.get((career_key(university), career_key(name)))
        warning, research, industry = [], [], []
        path, alignment, confidence = None, None, None
        method = 'rule_based_advisory'
        if found is None:
            warning.append('Program absent from catalog; no track or placement inference made.')
        else:
            # Program body only; the student's declared goal cannot become its own evidence.
            body = ' '.join(career_text(found.get(key)) for key in ('description_text', 'curriculum_highlights', 'faculty_research_areas'))
            research, industry = positive_mentions(body, RESEARCH_CUES), positive_mentions(body, INDUSTRY_CUES)
            supports = {'research': len(research), 'industry': len(industry)}
            if model_bundle is not None:
                method = 'supervised_text_classifier'
                estimator = model_bundle['model']
                probabilities = estimator.predict_proba([career_features(profile, program, found)])[0]
                classes = list(estimator.classes_)
                confidence = float(max(probabilities))
                if confidence >= model_bundle['decision_threshold']:
                    path = str(classes[int(np.argmax(probabilities))])
                else:
                    warning.append('Supervised model confidence below the development threshold.')
                if goal is not None:
                    alignment = float(probabilities[classes.index(goal)])
                warning.append('Classifier scores are not calibrated placement probabilities.')
            else:
                options = [track for track, count in supports.items() if count]
                if goal in options:
                    path = goal
                elif len(options) == 1:
                    path = options[0]
                if goal is not None and sum(supports.values()):
                    alignment = supports[goal] / sum(supports.values())
                if path is None:
                    warning.append('Insufficient or mixed evidence; confirm the career goal and program pathway.')
                warning.append('Advisory rule result; no outcome-trained model is available.')
        if path is not None and goal is not None and path != goal:
            warning.append('Program pathway evidence conflicts with the stated goal; review this mismatch.')
        roles, sources = suggested_roles(name, path) if found is not None else ([], [])
        country = career_text(program.get('country')) or (career_text(found.get('country')) if found else '')
        placements, placement_note = placement_for_program(placement_reports, university, name, country, as_of)
        if found is None:
            placements = None
        rows.append({'university': university, 'program_name': name, 'country': country or None,
            'career_path': path, 'alignment_score': alignment, 'expected_roles': roles,
            'avg_placement_stats': placements, 'method': method,
            'alignment_score_type': ('uncalibrated_goal_class_probability' if model_bundle else 'heuristic_goal_evidence_share'),
            'model_confidence': confidence, 'stated_goal': goal, 'goal_interpretation': goal_note,
            'track_evidence': {'research': research, 'industry': industry},
            'role_sources': sources, 'roles_note': 'Illustrative roles to explore, not predicted job offers; qualifications may vary.',
            'placement_note': placement_note, 'review_required': path is None or goal is None or path != goal,
            'warnings': warning})
    return {**state, 'careers': rows, 'status': 'career_done'}

print('Active entry point: career_agent(state). Output key: careers. Missing admission scores are tolerated.')

Active entry point: career_agent(state). Output key: careers. Missing admission scores are tolerated.


## 3. Future supervised classifier: real outcomes only

The framework proposes an outcome-trained research/industry classifier. This implementation
provides a TF-IDF + logistic-regression baseline, not a fine-tuned transformer. Features use
program text, interests and the stated goal; labels must be independently observed outcomes
or independently adjudicated track decisions, **never copied from the goal or these rules**.
Freeze feature snapshots at advising time; do not include post-outcome evidence.

Create a train/dev CSV and a separate sealed test CSV with columns:
`record_id,student_id,split,university_name,program_name,interests_text,career_goal,outcome_track,label_source,observed_on`.
Labels are `research`/`industry`; maintain the same definition across splits (research roles
may occur in academia or companies). Assign student groups once; train/dev needs at least
30 students and both classes. Thirty students is only a minimum validation guard, not enough
to support a 96% confidence claim. No automatic training or test execution occurs.

```python
# Only after independently labelled data exists:
bundle = train_career_classifier("path/to/train_dev.csv", independent_labels_confirmed=True)
print(bundle["dev_report"])  # development only
freeze_career_classifier(bundle, "v1")
test_report = evaluate_career_test_once(bundle, "path/to/sealed_test.csv", "v1")
state = career_agent(state, model_bundle=bundle)
```

The primary test accuracy counts abstentions as errors; selective accuracy and coverage are
reported separately. Report all test rows. Repeated students prevent an independent-query
Wilson claim. After test inspection/tuning, use a fresh student cohort. Local locks are audit
guards, not access control. Never commit private outcomes or per-student notebook outputs.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from datetime import datetime, timezone
import pickle

OUTCOME_COLUMNS = ['record_id', 'student_id', 'split', 'university_name', 'program_name',
                   'interests_text', 'career_goal', 'outcome_track', 'label_source', 'observed_on']

def load_career_outcomes(path, allowed_splits):
    data = pd.read_csv(path, dtype=str, keep_default_na=False)
    if data.empty or set(OUTCOME_COLUMNS) - set(data):
        raise ValueError('Real career outcomes with all schema fields are required; no generated labels.')
    for column in OUTCOME_COLUMNS:
        data[column] = data[column].str.strip()
        if column not in ('interests_text', 'career_goal') and data[column].eq('').any():
            raise ValueError('Missing outcome field: ' + column)
    if not set(data.split).issubset(set(allowed_splits)):
        raise ValueError('File includes a forbidden split. Keep train/dev and test physically separate.')
    if (data.record_id.duplicated().any() or data.groupby('student_id').split.nunique().gt(1).any()
            or not data.outcome_track.isin(['research', 'industry']).all()):
        raise ValueError('Duplicate records, overlapping students or invalid outcome labels.')
    dates = pd.to_datetime(data.observed_on, errors='raise', utc=True)
    if (dates > pd.Timestamp.now(tz='UTC')).any():
        raise ValueError('Future outcomes cannot be used as ground truth.')
    lookup = {(r['_university_key'], r['_program_key']): r for r in career_catalog.to_dict('records')}
    texts = []
    for row in data.to_dict('records'):
        key = (career_key(row['university_name']), career_key(row['program_name']))
        if key not in lookup:
            raise ValueError('Outcome program absent from current catalog; no rows dropped.')
        texts.append(career_features(row, row, lookup[key]))
    data['feature_text'] = texts
    return data

def train_career_classifier(train_dev_path, *, independent_labels_confirmed=False, decision_threshold=.60):
    if not independent_labels_confirmed:
        raise ValueError('Confirm consent and independently sourced real outcomes, never goal-derived labels.')
    if not .5 <= decision_threshold <= 1:
        raise ValueError('decision_threshold must be between .5 and 1 and chosen using dev only.')
    data = load_career_outcomes(train_dev_path, {'train', 'dev'})
    if set(data.split) != {'train', 'dev'} or data.student_id.nunique() < 30:
        raise ValueError('Need both train and dev with at least 30 independent students in total.')
    train, dev = data.loc[data.split.eq('train')], data.loc[data.split.eq('dev')]
    if any(part.outcome_track.nunique() != 2 for part in (train, dev)):
        raise ValueError('Both outcome classes must be represented in train and dev.')
    train_text = set(train.feature_text.map(career_key))
    if train_text & set(dev.feature_text.map(career_key)):
        raise ValueError('Duplicate model inputs across train/dev; group them before splitting.')
    model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True),
                          LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
    model.fit(train.feature_text, train.outcome_track)
    prediction = model.predict(dev.feature_text)
    report = {'evaluation_type': 'development_only', 'students': int(dev.student_id.nunique()),
        'records': len(dev), 'accuracy': float(accuracy_score(dev.outcome_track, prediction)),
        'balanced_accuracy': float(balanced_accuracy_score(dev.outcome_track, prediction)),
        'macro_f1': float(f1_score(dev.outcome_track, prediction, average='macro'))}
    return {'model': model, 'independent_labels_confirmed': True, 'decision_threshold': decision_threshold,
        'training_students': sorted(set(data.student_id)),
        'training_input_hashes': sorted({hashlib.sha256(career_key(text).encode()).hexdigest() for text in data.feature_text}),
        'training_data_sha256': hashlib.sha256(Path(train_dev_path).read_bytes()).hexdigest(), 'dev_report': report}

def career_model_fingerprint(bundle):
    # Both executable inference code and the fitted classifier/configuration are frozen.
    import marshal, types
    def stable_code(item):
        constants = tuple(stable_code(value) if isinstance(value, types.CodeType) else value for value in item.co_consts)
        return item.replace(co_filename='<career-evaluation>', co_firstlineno=1, co_consts=constants)
    digest = hashlib.sha256(pickle.dumps(bundle, protocol=5))
    for function in (career_agent, career_features, stated_career_goal, career_text, career_key, positive_mentions):
        digest.update(marshal.dumps(stable_code(function.__code__)))
    digest.update(json.dumps([RESEARCH_CUES, INDUSTRY_CUES, GOAL_RESEARCH, GOAL_INDUSTRY]).encode())
    digest.update(career_catalog.to_json(orient='records').encode())
    return digest.hexdigest()

def freeze_career_classifier(bundle, version):
    if not re.fullmatch(r'[A-Za-z0-9_-]+', version) or not bundle.get('independent_labels_confirmed'):
        raise ValueError('Use a unique simple version name and a verified outcome-trained bundle.')
    AGENT6_RUNS.mkdir(parents=True, exist_ok=True)
    with (AGENT6_RUNS / f'{version}.freeze.json').open('x', encoding='utf-8') as handle:
        json.dump({'version': version, 'fingerprint': career_model_fingerprint(bundle),
                   'frozen_at': datetime.now(timezone.utc).isoformat()}, handle, indent=2)

def evaluate_career_test_once(bundle, test_path, version):
    freeze = json.loads((AGENT6_RUNS / f'{version}.freeze.json').read_text(encoding='utf-8'))
    if freeze['fingerprint'] != career_model_fingerprint(bundle):
        raise ValueError('Classifier/catalog changed after freezing.')
    test = load_career_outcomes(test_path, {'test'})
    if set(test.student_id) & set(bundle['training_students']):
        raise ValueError('Test students overlap train/dev.')
    test_hashes = {hashlib.sha256(career_key(text).encode()).hexdigest() for text in test.feature_text}
    if test_hashes & set(bundle['training_input_hashes']):
        raise ValueError('Test model inputs duplicate train/dev inputs.')
    for path in AGENT6_RUNS.glob('*.consumed.json'):
        if set(test.student_id) & set(json.loads(path.read_text(encoding='utf-8'))['student_ids']):
            raise ValueError('Test cohort already consumed; collect fresh students after tuning.')
    with (AGENT6_RUNS / f'{version}.consumed.json').open('x', encoding='utf-8') as handle:
        json.dump({'student_ids': sorted(set(test.student_id)), 'test_sha256': hashlib.sha256(Path(test_path).read_bytes()).hexdigest()}, handle)
    prediction, confidence = [], []
    # Test the same public career_agent used by the pipeline, including abstention.
    for row in test.to_dict('records'):
        state = {'profile': row, 'programs': [{'university': row['university_name'], 'program_name': row['program_name']}]}
        result = career_agent(state, model_bundle=bundle)['careers'][0]
        prediction.append(result['career_path'] or 'unknown')
        confidence.append(result['model_confidence'])
    predicted = np.array(prediction)
    decided = predicted != 'unknown'
    correct = predicted == test.outcome_track.to_numpy()
    count = int(decided.sum())
    overall = float(correct.mean())  # abstentions count as incorrect in this primary metric
    z, n = 1.959963984540054, len(test)
    centre = (overall + z*z/(2*n))/(1+z*z/n)
    radius = z*math.sqrt(overall*(1-overall)/n + z*z/(4*n*n))/(1+z*z/n)
    independent = test.student_id.nunique() == n
    report = {'version': version, 'fingerprint': freeze['fingerprint'], 'test_records': n,
        'test_students': int(test.student_id.nunique()), 'accuracy_including_abstentions': overall,
        'decision_coverage': float(decided.mean()), 'unknown_share': float((~decided).mean()),
        'selective_accuracy': float(correct[decided].mean()) if count else None,
        'accuracy_wilson_95': [max(0, centre-radius), min(1, centre+radius)],
        'wilson_independent_students': bool(independent),
        'target_96_supported': bool(independent and centre-radius >= .96),
        'confusion_matrix': pd.crosstab(test.outcome_track, pd.Series(prediction, index=test.index)).to_dict()}
    with (AGENT6_RUNS / f'{version}.report.json').open('x', encoding='utf-8') as handle:
        json.dump(report, handle, indent=2)
    test.assign(prediction=prediction, confidence=confidence).to_json(
        AGENT6_RUNS / f'{version}.predictions.json', orient='records', indent=2)
    return report

career_model_bundle = None
print('Supervised training is available explicitly, but no model has been trained on invented labels.')

Supervised training is available explicitly, but no model has been trained on invented labels.


## 4. Execute the actual Agent 3 -> Agent 6 handoff

The following demonstration calls the real `program_agent` and then `career_agent` with
real catalog entries. Its profile and shortlist are illustrative, not real student test data.
The full-catalog run measures processing coverage only, not career-outcome accuracy.

In [4]:
# Import only Agent 3's actual setup/implementation cell into an isolated namespace.
# This does not read its judgments, run its diagnostics, or consume any frozen test cohort.
import contextlib
import io
agent3_path = PROJECT_ROOT / 'backend/app/agents/program_agent/Agent3_Program_agent.ipynb'
agent3_notebook = json.loads(agent3_path.read_text(encoding='utf-8'))
setup = next(''.join(c['source']) for c in agent3_notebook['cells']
             if c['cell_type'] == 'code' and 'class ProgramRankerV2:' in ''.join(c['source']))
agent3_kernel = {}
with contextlib.redirect_stdout(io.StringIO()):
    exec(compile(setup, 'Agent3_implementation_for_career_handoff', 'exec'), agent3_kernel)

demo_profile = {'interests_text': 'distributed systems, cloud infrastructure and backend engineering',
    'target_program': 'Computer Science', 'target_degree': 'MS', 'career_goal': 'industry',
    'gpa': 8.6, 'gpa_scale': 10, 'profile_strength_score': 72}
demo_shortlist = {'profile': demo_profile, 'matched_universities': [
    {'university': 'Stanford University', 'country': 'United States', 'programs': [
        {'program_name': 'MS Computer Science', 'match_score': .787, 'admit_probability': .566, 'tier': 'Target'},
        {'program_name': 'MS Computer Science - Systems', 'match_score': .793, 'admit_probability': .566, 'tier': 'Target'}]}]}
agent3_live_demo = agent3_kernel['program_agent'](demo_shortlist)
demo_original = deepcopy(agent3_live_demo)
agent6_live_demo = career_agent(agent3_live_demo)
assert agent3_live_demo == demo_original
assert [(r['university'], r['program_name']) for r in agent6_live_demo['careers']] == [
    (r['university'], r['program_name']) for r in agent3_live_demo['programs']]
assert all(r['avg_placement_stats'] is None for r in agent6_live_demo['careers'])
assert agent6_live_demo['careers'][0]['program_name'] == 'MS Computer Science - Systems'
print('REAL Agent 3 -> Agent 6 function handoff; illustrative student and shortlist, real program catalog:')
print(pd.DataFrame(agent6_live_demo['careers'])[['university', 'program_name', 'career_path',
    'alignment_score', 'expected_roles', 'method']].to_string(index=False))
print('Placement statistics: unknown (no source report supplied). Alignment scores are heuristic evidence shares.')

# Coverage check across all real program identities; no career-outcome labels are inferred.
coverage_state = {'profile': {'career_goal': 'industry'}, 'programs': [
    {'university': r['university_name'], 'program_name': r['program_name'], 'country': r.get('country')}
    for r in career_catalog.to_dict('records')]}
career_catalog_result = career_agent(coverage_state)
career_catalog_counts = {'input_programs': len(coverage_state['programs']),
    'output_programs': len(career_catalog_result['careers']),
    'track_decisions': sum(r['career_path'] is not None for r in career_catalog_result['careers']),
    'with_role_examples': sum(bool(r['expected_roles']) for r in career_catalog_result['careers']),
    'with_verified_placement_stats': sum(r['avg_placement_stats'] is not None for r in career_catalog_result['careers'])}
assert career_catalog_counts['input_programs'] == career_catalog_counts['output_programs']
print('Catalog execution coverage (not accuracy):', career_catalog_counts)

REAL Agent 3 -> Agent 6 function handoff; illustrative student and shortlist, real program catalog:
         university                  program_name career_path  alignment_score       expected_roles              method
Stanford University MS Computer Science - Systems    industry         0.666667 [Software developer] rule_based_advisory
Stanford University           MS Computer Science    industry         0.666667 [Software developer] rule_based_advisory
Placement statistics: unknown (no source report supplied). Alignment scores are heuristic evidence shares.
Catalog execution coverage (not accuracy): {'input_programs': 3052, 'output_programs': 3052, 'track_decisions': 3052, 'with_role_examples': 3052, 'with_verified_placement_stats': 0}


## 5. Executed functional prediction checks — explicitly fictional

In [5]:
# Fictional examples independently specify expected rule behavior, not real career labels.
fixture_career_catalog = pd.DataFrame([
    {'university_name': 'Fixture University', 'program_name': 'Research Computing',
     'description_text': 'Thesis and dissertation research with experiments.', 'curriculum_highlights': '', 'faculty_research_areas': '', 'country': 'United States'},
    {'university_name': 'Fixture University', 'program_name': 'Professional Computing',
     'description_text': 'Professional software design, industry internship and capstone.', 'curriculum_highlights': '', 'faculty_research_areas': '', 'country': 'United States'},
    {'university_name': 'Fixture University', 'program_name': 'Mixed Computing',
     'description_text': 'Research thesis plus industry internship and software design.', 'curriculum_highlights': '', 'faculty_research_areas': '', 'country': 'United States'},
    {'university_name': 'Fixture University', 'program_name': 'No Evidence',
     'description_text': '', 'curriculum_highlights': '', 'faculty_research_areas': '', 'country': 'United States'},
    {'university_name': 'Fixture University', 'program_name': 'Negated Research',
     'description_text': 'No thesis. No research. Industry internship and professional capstone.', 'curriculum_highlights': '', 'faculty_research_areas': '', 'country': 'United States'},
])
fixture_career_catalog['_university_key'] = fixture_career_catalog.university_name.map(career_key)
fixture_career_catalog['_program_key'] = fixture_career_catalog.program_name.map(career_key)
career_cases = [
    ('Research goal and research evidence', 'Research Computing', {'career_goal': 'research'}, 'research'),
    ('Industry goal and professional evidence', 'Professional Computing', {'career_goal': 'industry'}, 'industry'),
    ('Mixed program honors research goal', 'Mixed Computing', {'career_goal': 'research'}, 'research'),
    ('Mixed program honors industry goal', 'Mixed Computing', {'career_goal': 'industry'}, 'industry'),
    ('Missing goal, mixed program abstains', 'Mixed Computing', {}, None),
    ('No descriptive evidence abstains', 'No Evidence', {'career_goal': 'research'}, None),
    ('Unknown program abstains', 'Absent Program', {'career_goal': 'industry'}, None),
    ('Opposite-only evidence flags mismatch', 'Professional Computing', {'career_goal': 'research'}, 'industry'),
    ('Negated research is not positive evidence', 'Negated Research', {'career_goal': 'industry'}, 'industry'),
    ('Negated free-text research goal', 'Mixed Computing', {'career_goals': 'I do not want research; I want an industry job.'}, 'industry'),
    ('Mixed free-text goal abstains', 'Mixed Computing', {'career_goals': 'Research and industry both interest me'}, None),
    ('Interests alone do not invent a goal', 'Mixed Computing', {'interests_text': 'research scientist'}, None),
]
career_check_rows = []
for name, program, profile, expected in career_cases:
    state = {'profile': profile, 'programs': [{'university': 'Fixture University', 'program_name': program,
                                              'admit_probability': None, 'tier': 'Unknown'}]}
    original = deepcopy(state)
    row = career_agent(state, fixture_career_catalog)['careers'][0]
    passed = row['career_path'] == expected and original == state and row['avg_placement_stats'] is None
    if name == 'Opposite-only evidence flags mismatch':
        passed = passed and row['alignment_score'] == 0
    career_check_rows.append({'case': name, 'expected': expected, 'predicted': row['career_path'], 'passed': passed})
assert career_agent({'profile': {}})['careers'] == []
career_check_rows.append({'case': 'Empty Agent 3 shortlist', 'expected': 'empty', 'predicted': 'empty', 'passed': True})

# A source-backed rate is taken from one exact cohort, never from unrelated job statistics.
placement_fixture = pd.DataFrame([{'university_name': 'Fixture University', 'program_name': 'Professional Computing',
    'cohort_year': 2025, 'country': 'United States', 'definition': 'Fictional fixture: employed within six months among respondents',
    'placed_count': 80, 'surveyed_count': 100, 'source_url': 'https://example.org/fictional-fixture', 'verified_on': '2026-01-01'}])
fixture_program_state = {'profile': {'career_goal': 'industry'}, 'programs': [
    {'university': 'Fixture University', 'program_name': 'Professional Computing'}]}
row = career_agent(fixture_program_state, fixture_career_catalog, placement_reports=placement_fixture, as_of='2026-10-21')['careers'][0]
career_check_rows.append({'case': 'Explicit cohort counts compute rate', 'expected': .8,
                         'predicted': row['avg_placement_stats']['placement_rate'], 'passed': row['avg_placement_stats']['placement_rate'] == .8})
invalid_report = placement_fixture.copy()
invalid_report['placed_count'] = 120
row = career_agent(fixture_program_state, fixture_career_catalog, placement_reports=invalid_report, as_of='2026-10-21')['careers'][0]
career_check_rows.append({'case': 'Impossible placement counts withheld', 'expected': None,
                         'predicted': row['avg_placement_stats'], 'passed': row['avg_placement_stats'] is None})
career_functional_checks = pd.DataFrame(career_check_rows)
print(career_functional_checks.to_string(index=False))
assert career_functional_checks.passed.all(), 'A displayed career contract case failed.'
print('Functional checks only; these fixture percentages are not outcome-classifier accuracy.')

                                     case expected predicted  passed
      Research goal and research evidence research  research    True
  Industry goal and professional evidence industry  industry    True
       Mixed program honors research goal research  research    True
       Mixed program honors industry goal industry  industry    True
     Missing goal, mixed program abstains     None      None    True
         No descriptive evidence abstains     None      None    True
                 Unknown program abstains     None      None    True
    Opposite-only evidence flags mismatch industry  industry    True
Negated research is not positive evidence industry  industry    True
          Negated free-text research goal industry  industry    True
            Mixed free-text goal abstains     None      None    True
     Interests alone do not invent a goal     None      None    True
                  Empty Agent 3 shortlist    empty     empty    True
      Explicit cohort counts compu

## 6. Use in the existing pipeline

```python
state = program_agent(state)
state = career_agent(state)  # adds careers; retains programs, finance and scholarships
```

Future placement report schema: `university_name,program_name,cohort_year,country,definition,placed_count,surveyed_count,source_url,verified_on`.
Load a reviewed DataFrame and pass `placement_reports=reports`. URLs and dates record human
provenance; their presence alone cannot prove that a report was verified.

## Final executed results

In [6]:
print('AGENT 6 - FINAL EXECUTED RESULTS')
print('Method: rule-based advisory fallback; no labelled career-outcome dataset supplied.')
print(f'Functional prediction checks: {int(career_functional_checks.passed.sum())}/{len(career_functional_checks)} '
      f'passed ({career_functional_checks.passed.mean():.2%}). This is a check pass rate, not predictive accuracy.')
print('Agent 3 -> Agent 6 handoff: PASSED; Agent 3 order and caller state preserved.')
print('Real-catalog coverage:', career_catalog_counts)
saved_career_reports = sorted(AGENT6_RUNS.glob('*.report.json'), key=lambda p: p.stat().st_mtime)
if saved_career_reports:
    saved_career_test = json.loads(saved_career_reports[-1].read_text(encoding='utf-8'))
    print('Saved supervised test report (not rerun; separate from the active rule fallback):', saved_career_test)
else:
    print('Independent research/industry classification accuracy: NOT MEASURED.')
    print('96% real-world accuracy claim: NOT VERIFIED.')
print('Placement statistics: NOT AVAILABLE without source-backed program/cohort reports.')
print('Example career predictions:')
print(pd.DataFrame(agent6_live_demo['careers'])[['program_name', 'career_path', 'expected_roles', 'alignment_score']].to_string(index=False))

AGENT 6 - FINAL EXECUTED RESULTS
Method: rule-based advisory fallback; no labelled career-outcome dataset supplied.
Functional prediction checks: 15/15 passed (100.00%). This is a check pass rate, not predictive accuracy.
Agent 3 -> Agent 6 handoff: PASSED; Agent 3 order and caller state preserved.
Real-catalog coverage: {'input_programs': 3052, 'output_programs': 3052, 'track_decisions': 3052, 'with_role_examples': 3052, 'with_verified_placement_stats': 0}
Independent research/industry classification accuracy: NOT MEASURED.
96% real-world accuracy claim: NOT VERIFIED.
Placement statistics: NOT AVAILABLE without source-backed program/cohort reports.
Example career predictions:
                 program_name career_path       expected_roles  alignment_score
MS Computer Science - Systems    industry [Software developer]         0.666667
          MS Computer Science    industry [Software developer]         0.666667
